In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain_cohere import CohereRerank
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from dotenv import load_dotenv

load_dotenv()

import langchain
print(langchain.__version__)

import os

1.3.1


In [65]:
llm = OllamaLLM(model="llama3.2")

embeddings_model = OllamaEmbeddings(model="llama3.2")

In [66]:
# Carregar o pdf
pdf_link="os-sertoes.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [67]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [68]:
vectorstore = Chroma.from_documents(
    chunks, 
    embedding=embeddings_model,
    persist_directory="sertoes_rerank_rag"
)

In [69]:
# Load Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 10
    }
)

In [70]:
rerank = CohereRerank(top_n=3, model="rerank-v3.5")

compressor_retriver = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=retriever
)

In [71]:
prompt = ChatPromptTemplate.from_template("""
    Responda a pergunta com base apenas no contexto abaixo:
    
    <context>
    {context}
    </context>
    
    Pergunta: {question}
""")

In [72]:
setup_retrival = RunnableParallel({"question": RunnablePassthrough(), "context": compressor_retriver})

output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrival | prompt | llm | output_parser

In [73]:
def ask(question):
    docs = compressor_retriver.invoke(question)
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---")
        print(doc.page_content)
        print(f"Metadata: {doc.metadata}")
        print()
    
    return compressor_retrieval_chain.invoke(question)

In [74]:
user_question = input("User: ")
answer = ask(user_question)
print("Answer: ", answer)

User:  Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?


--- Chunk 1 ---
os sertões | euclides da cunha
273
Estava conquistada a montanha após três horas de conflito. A 
vitória, porém, resultava da coragem cega junta à mais completa 
indisciplina de fogo – e compreende-se que mais tarde a ordem 
do dia relativa ao feito desse preeminente lugar às praças gradua -
das. Os seus cabos de guerra foram os cabos de esquadra. Sobre 
os jagunços em fuga confluíram cargas em desordem: soldados 
em grupos, turbas sem comando, disparando à toa as carabinas, 
num fanfarrear irritante e numa alacridade feroz de monteiros no 
último lance de uma batida a javardos.
Os jagunços escapavam-se-lhes adiante. Perseguiram-nos.
A artilharia, embaixo, começou a rodar, puxada a pulso, pelas 
ladeiras acima.
Realizara-se a travessia; e, tirante o dispêndio de munições, 
eram poucas as perdas – quatro mortos e vinte e tantos feridos. 
Em troca os sertanejos deixavam cento e quinze cadáveres, conta -
dos rigorosamente.
Episódio trágico 
Fora uma hecatombe. Cumulou-a um